In [0]:
%sql
select * from novacart.novacart_schema.products limit 5;

# Step 1-Imports and setups

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timedelta
import uuid


In [0]:

spark.sql("create schema if not exists datatocrunch_novacart.bronze_schema")

#Step 2 - Bronze control table

This table stores the watermark for each source table.
It helps the pipeline remember:
- the latest timestamp already processed
- the latest primary key processed at that timestamp
- how many rows were written in last run

### This table type ingestion_control is delta table

In [0]:
spark.sql("""
          create table if not exists datatocrunch_novacart.bronze_schema.ingestion_control(
              medallion_layer string,
              table_name string,
              watermarkColName_of_table string,
              pkColName_of_table string,
              last_successful_ts timestamp,
              last_successful_pk bigint,
              last_successful_batch_id string,
              rows_written bigint,
              updated_ts timestamp,
              run_status string
          )
          using delta
          """)

In [0]:
%sql
--select * from datatocrunch_novacart.bronze_schema.ingestion_control

--drop table if exists datatocrunch_novacart.bronze_schema.ingestion_control;

In [0]:
%sql
ALTER TABLE datatocrunch_novacart.bronze_schema.ingestion_control SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

In [0]:
%sql
--alter table datatocrunch_novacart.bronze_schema.ingestion_control rename column curr_ts to watermark;

In [0]:
%sql
--ALTER TABLE datatocrunch_novacart.bronze_schema.ingestion_control ALTER COLUMN watermark TYPE STRING;

In [0]:
%sql
--alter table datatocrunch_novacart.bronze_schema.ingestion_control add column run_status string;

#Step 3 - Source table configuration

This cell defines which source tables will be loaded into Bronze and which columns should be used as:

- primary key
- timestamp/watermark column

It also creates a unique bronze_run_id for the current pipeline run.


In [0]:

table_config={
    "orders":{"pkColName_of_table":"order_id","watermarkColName_of_table":"updated_at"},
    "products":{"pkColName_of_table":"product_id","watermarkColName_of_table":"updated_at"},
    "payments":{"pkColName_of_table":"payment_id","watermarkColName_of_table":"processed_at"}
}
bronze_runid=str(uuid.uuid4())
print("Current bronze runid: "+bronze_runid)

#Step-4 User defined functions
This cell contains reusable functions:
- get_last_successful_watermark() reads the last processed watermark from the control table
- upsert_bronze_control() updates the control table after a successful Bronze load

These functions keep the main load logic cleaner and easier to understand

In [0]:
from pyspark.sql import functions as F

def get_last_successful_watermark(table_name: str):

    res = (
        spark.table("datatocrunch_novacart.bronze_schema.ingestion_control")
        .filter(
            (F.col("medallion_layer") == "bronze") &
            (F.col("table_name") == table_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("last_successful_ts").desc())
        .limit(1)
        .collect()
    )

    if len(res) == 0:
        return None, None

    return (
        res[0]["last_successful_ts"],
        res[0]["last_successful_pk"]
    )

In [0]:
def upsert_bronze_control(table_name,watermarkColName_of_table,pkColName_of_table,last_successful_ts,last_successful_pk,last_successful_batch_id,rows_written,updated_ts):
    df=spark.createDataFrame(
        [{
            "medallion_layer":"bronze",
            "table_name":table_name,
            "watermarkColName_of_table":watermarkColName_of_table,
            "pkColName_of_table":pkColName_of_table,
            "last_successful_ts":last_successful_ts,
            "last_successful_pk":last_successful_pk,
            "last_successful_batch_id":last_successful_batch_id,
            "rows_written":rows_written,
            "run_status":"success",
            "updated_ts":datetime.now()
        }]
    )

    target=DeltaTable.forName(spark,"datatocrunch_novacart.bronze_schema.ingestion_control")

    (target.alias("t")
     .merge(df.alias("s"), """
            t.table_name = s.table_name
            AND t.medallion_layer = s.medallion_layer
            """)
     .whenMatchedUpdate(
         set={
            "medallion_layer": "s.medallion_layer",
            "table_name": "s.table_name",
            "watermarkColName_of_table": "s.watermarkColName_of_table",
            "pkColName_of_table": "s.pkColName_of_table",
            "last_successful_ts": "s.last_successful_ts",
            "last_successful_pk": "s.last_successful_pk",
            "last_successful_batch_id": "s.last_successful_batch_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_ts": "s.updated_ts"
         }
     )
     .whenNotMatchedInsertAll()
     .execute() 
     )
    


#Step 5 - Bronze incremental load loop

This is the main Bronze logic.

For each table, the notebook:

1. reads the last watermark

2. reads the source SQL table

3. filters only new/changed rows

4. adds Bronze audit columns

5. appends the rows into the Bronze Delta table

6. updates the control table

This is the core incremental loading logic.

In [0]:
for key, val in table_config.items():

    pkColName_of_table = val["pkColName_of_table"]
    watermarkColName_of_table = val["watermarkColName_of_table"]

    source_table = f"novacart.novacart_schema.{key}"
    target_table = f"datatocrunch_novacart.bronze_schema.{key}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(key)

    source_df = spark.read.table(source_table).withColumn(watermarkColName_of_table,F.col(watermarkColName_of_table).cast("timestamp"))

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(watermarkColName_of_table) > F.lit(last_successful_ts))
            |
            (
                (F.col(watermarkColName_of_table) == F.lit(last_successful_ts))
                &
                (F.col(pkColName_of_table) > F.lit(last_successful_pk))
            )
        )

    #Audit    
    rows_to_load = (
    rows_to_load
    .withColumn("bronze_ingested_at", F.current_timestamp())
    .withColumn("bronze_runid", F.lit(bronze_runid))
    .withColumn("bronze_source_table", F.lit(source_table))
)
    row_count = rows_to_load.count()

    if row_count == 0:
        print(f"No new records found for {key}")
        rows_to_load.unpersist()
        continue

    rows_to_load.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

    max_ts = rows_to_load.agg(
        F.max(watermarkColName_of_table).alias("max_ts")
    ).collect()[0]["max_ts"]

    max_pk = (
        rows_to_load
        .filter(F.col(watermarkColName_of_table) == max_ts)
        .agg(
            F.max(F.col(pkColName_of_table).cast("long")).alias("max_pk")
        )
        .collect()[0]["max_pk"]
    )
    last_successful_batch_id=bronze_runid

    upsert_bronze_control(
        key,
        watermarkColName_of_table,
        pkColName_of_table,
        max_ts,
        max_pk,
        last_successful_batch_id,
        row_count,
        datetime.now()
    )

 

In [0]:
spark.table(
    "datatocrunch_novacart.bronze_schema.ingestion_control"
).printSchema()

In [0]:
%sql
select * from datatocrunch_novacart.bronze_schema.ingestion_control

In [0]:
%sql
--truncate table datatocrunch_novacart.bronze_schema.ingestion_control

In [0]:
%sql
select * from datatocrunch_novacart.bronze_schema.orders_raw limit 5;